# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists('FlyRank_AI'):
    !git clone -q https://github.com/AhmedMahmoud-123/FlyRank_AI.git
os.chdir('FlyRank_AI')
!python scripts/01_prepare_features.py

import sys
sys.path.append('scripts')
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv('data/processed/refresh_feature_vector.csv')
print(f'{len(df):,} rows loaded')

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank_AI/data/processed/refresh_feature_vector.csv
30,000 rows loaded


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [2]:
feature_cols = ['impressions_prev_30d', 'avg_position', 'days_since_last_update', 'has_clicks']
model_data = df.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining_label']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_data['client_id']))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
model_data = model_data.copy()
model_data.loc[X.index, 'risk_score'] = rf.predict_proba(X)[:, 1]

def reason_code(row):
    if row['days_since_last_update'] >= 180 and row['impressions_prev_30d'] >= 500:
        return 'stale_but_visible'
    if row['avg_position'] > model_data.loc[model_data['avg_position'] > 0, 'avg_position'].median():
        return 'position_slipping'
    return 'low_signal'

model_data['reason_code'] = model_data.apply(reason_code, axis=1)
queue = model_data.sort_values('risk_score', ascending=False)

print(f'base rate: {y.mean():.3f}')
queue[['content_id', 'risk_score', 'reason_code', 'impressions_prev_30d', 'avg_position', 'days_since_last_update']].head(20)

base rate: 0.542


,content_id,risk_score,reason_code,impressions_prev_30d,avg_position,days_since_last_update
14413,content_0636f7bcfa05,1.0,position_slipping,4346,24.1,26
28492,content_15822f1838f6,1.0,low_signal,35,6.8,104
4042,content_43dafa0d1f03,1.0,low_signal,6,7.3,8
4041,content_0cfb18d748e2,1.0,position_slipping,1,15.5,8
19609,content_35740b28d1ce,1.0,low_signal,27,8.1,8
19608,content_eb343718a4e4,1.0,position_slipping,1041,18.2,14
21783,content_135baf84d5ce,1.0,low_signal,3,2.1,20
17330,content_ab11e71992b7,1.0,low_signal,154,8.5,8
28503,content_8d7c1d704b43,1.0,low_signal,31,3.1,20
11753,content_b854e1b19fe8,1.0,low_signal,63,3.5,20


**Ranked actions:** each row is scored by predicted decline risk, with a plain-language reason
code attached — `stale_but_visible` (real traffic, untouched a long time) or `position_slipping`
(worse-than-median position). A person reading row 1 should understand *why* it's row 1 without
reading any code, per the baseline's transparency standard from Week 4.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
print(f"Queue covers {len(queue):,} content items, model trained on {len(X_tr):,} rows "
      f"from {model_data.iloc[train_idx]['client_id'].nunique()} clients, "
      f"tested on {model_data.iloc[test_idx]['client_id'].nunique()} held-out clients.")

**Intended use:** a content strategist's starting point for weekly review — which pages to
look at first, not a final verdict. Built for the Refresh / Content Opportunity lane specifically;
not validated for other content types or other clients' data outside this dataset.

**Limits (claim-ladder honest):**
- This is a **decision-support ranking**, not a prediction of what will happen to any single
  page — cross-sectional data from one dataset snapshot cannot support "refreshing this page
  will fix it."
- Validated on a client-held-out split (see w06), but only within *this* dataset's client mix —
  it hasn't been tested on a genuinely new client's data.
- The leakage audit (w03/w06) removed features that inflated earlier accuracy numbers — this
  queue's risk scores are the honest, post-audit version, not the higher pre-audit numbers.
- Base rate is [Y]% — a risk score needs to clear that bar meaningfully to be worth acting on;
  a score only slightly above the base rate isn't a strong signal.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# Flag rows where the model and the simple baseline rule disagree -- worth a human's attention
visible = (model_data['impressions_prev_30d'] >= 500).astype(int)
stale = (model_data['days_since_last_update'] >= 180).astype(int)
baseline_flag = (visible & stale).astype(int)
model_flag = (model_data['risk_score'] >= 0.5).astype(int)

disagreement = model_data[baseline_flag != model_flag]
print(f'{len(disagreement):,} rows where model and baseline rule disagree -- review these first')
disagreement[['content_id', 'risk_score', 'reason_code']].head(10)

**What a person must check before acting:**
- Rows where the model and Week-4 baseline rule disagree (above) — either could be right;
  a human should look at the actual page, not just trust the higher-tech option by default.
- Any row with a reason code but very low `impressions_prev_30d` (near-zero traffic) — the
  model may be picking up noise on a page nobody's really looking at anyway.

**No-go list — never automate:**
- Do not auto-publish or auto-edit content based on this score alone; it flags candidates for
  a human writer/editor, nothing more.
- Do not use this ranking to make client-facing promises about traffic outcomes — no causal
  claim is supported (per writing-honest-claims: cross-sectional data, no intervention design).
- Do not apply this model to a client or content type outside the Refresh lane without
  re-validating — the honest-split numbers here are specific to this data.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Signs this playbook has gone stale:**
- The gap between random-split and grouped-split accuracy (measured in w06) widens over time —
  suggests the model is increasingly leaning on client-specific patterns again as new clients
  join the dataset.
- The base rate of `is_declining_label` shifts meaningfully from the [Y]% measured here —
  a changed base rate means the risk scores need recalibrating, not just reusing.
- Reason codes stop matching what reviewers find on inspection (tracked via Section 3's
  disagreement log) — rising disagreement rate is an early warning before accuracy visibly drops.
- **Retrain trigger:** re-run w06's leakage-and-split audit whenever the raw data refreshes
  (new month of `content_refresh_anonymized.csv`), not just when accuracy looks off — leakage
  can return silently if the feature-prep script changes.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [8]:
Path('work/outputs').mkdir(parents=True, exist_ok=True)
export_cols = ['content_id', 'risk_score', 'reason_code', 'impressions_prev_30d',
               'avg_position', 'days_since_last_update', 'is_declining_label']
queue[export_cols].to_csv('work/outputs/action_playbook_queue.csv', index=False)

summary = {
    'base_rate': float(y.mean()),
    'n_scored': int(len(queue)),
    'n_clients_train': int(model_data.iloc[train_idx]['client_id'].nunique()),
    'n_clients_test': int(model_data.iloc[test_idx]['client_id'].nunique()),
    'n_disagreements_with_baseline': int(len(disagreement)),
}
import json
with open('work/outputs/playbook_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Wrote work/outputs/action_playbook_queue.csv and playbook_summary.json")
print(summary)

Wrote work/outputs/action_playbook_queue.csv and playbook_summary.json
{'base_rate': 0.5420666666666667, 'n_scored': 30000, 'n_clients_train': 24, 'n_clients_test': 8, 'n_disagreements_with_baseline': 17245}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.